## Test the Libraries

In [ ]:
import os
import sys
import subprocess
import textwrap

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["JAX_PLATFORMS"] = "cpu"

WIDTH = 15
failed = []


def run_subprocess_test(name, code):
    proc = subprocess.run(
        [sys.executable, "-c", code],
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,   # ← key: hide native spam
        text=True,
    )

    if proc.returncode == 0:
        print(f"[ OK ] {name:<{WIDTH}} | package works properly")
    else:
        print(f"[FAIL] {name:<{WIDTH}} | subprocess failed")
        failed.append(name)


tests = {
    "pandas": """
import pandas as pd
df = pd.DataFrame({"a":[1,2,3]})
assert df["a"].mean()==2
""",
    "scikit-learn": """
from sklearn.linear_model import LinearRegression
import numpy as np
X=np.array([[1],[2],[3]])
y=np.array([2,4,6])
LinearRegression().fit(X,y)
""",
    "sympy": """
import sympy as sp
x=sp.symbols('x')
assert sp.diff(x**2,x)==2*x
""",
    "matplotlib": """
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.plot([1,2],[1,2])
""",
    "seaborn": """
import matplotlib
matplotlib.use("Agg")
import seaborn as sns, pandas as pd
sns.lineplot(data=pd.DataFrame({'x':[1,2],'y':[1,2]}),x='x',y='y')
""",
    "jax": """
import jax.numpy as jnp
jnp.sin(jnp.array([1.0]))
""",
    "mpi4py": """
from mpi4py import MPI
MPI.COMM_WORLD.Get_size()
""",
    "keras": """
from keras.models import Sequential
from keras.layers import Dense,Input
Sequential([Input(shape=(1,)),Dense(1)])
""",
    "tensorflow": """
import tensorflow as tf
tf.reduce_sum(tf.constant([1.0,2.0]))
""",
    "pytorch": """
import torch
torch.sum(torch.tensor([1.0,2.0]))
""",
}

print("\n===== Testing Python environment =====\n")

for name, code in tests.items():
    run_subprocess_test(name, textwrap.dedent(code))

print("\n===== Summary =====")

if not failed:
    print("All packages work properly ✅")
else:
    print("Failing packages:", ", ".join(failed))
    sys.exit(1)